# Analyse exploratoire : la dépression chez les étudiants

Dans ce notebook, j'explore le **Student Depression Dataset** (Kaggle), qui décrit 4 464 étudiants et indique s'ils déclarent être en dépression.

L'objectif n'est pas encore de prédire quoi que ce soit, mais de comprendre les données : leur structure, leur qualité, et comment le taux de dépression varie selon le profil des étudiants.

Outils utilisés : pandas et numpy.

In [1]:
import numpy as np
import pandas as pd

## Partie 1 — Découvrir les données

In [2]:
data = pd.read_csv("../data/Student_Depression_Dataset.csv")
data.head()  # j'affiche les 5 premières lignes pour voir à quoi ressemble le DataFrame

,Gender,Age,Family_background,Relationship_status,Marital_status,Living_style,Current_semester,CGPA,friend_circle,Health_condition,Cost_without_study,Home_division,Depression
0,Female,22,Middle class,Single,Unmarried,Bachelor,8,3.43,Yes,Critical,10113,Chittagong,Yes
1,Male,19,Middle class,Single,Unmarried,Bachelor,2,3.70,Yes,Abnormal,13815,Chittagong,Yes
2,Male,23,Middle class,Single,Unmarried,Family,8,3.28,No,Abnormal,15450,Dhaka,Yes
3,Male,20,Middle class,Complicated,Unmarried,Family,2,3.77,No,Good,17413,Dhaka,No
4,Female,21,Middle class,Single,Unmarried,Family,1,4.00,No,Abnormal,16693,Dhaka,Yes


In [3]:
data.shape

(4464, 13)

Le dataset contient 4 464 lignes (une par étudiant) et 13 colonnes.

In [4]:
data.dtypes

Gender                     str
Age                      int64
Family_background          str
Relationship_status        str
Marital_status             str
Living_style               str
Current_semester         int64
CGPA                   float64
friend_circle              str
Health_condition           str
Cost_without_study       int64
Home_division              str
Depression                 str
dtype: object

Les colonnes en int64 et float64 sont numériques (`Age`, `Current_semester`, `CGPA`, `Cost_without_study`), les autres contiennent du texte.

In [5]:
print("Lignes en double :", data.duplicated().sum())
print("Valeurs manquantes :", data.isna().sum().sum())

Lignes en double : 0
Valeurs manquantes : 0


Il n'y a aucune ligne en double et aucune valeur manquante.

In [6]:
colonnes_texte = ["Gender", "Family_background", "Relationship_status", "Marital_status",
                  "Living_style", "friend_circle", "Health_condition", "Home_division", "Depression"]

for colonne in colonnes_texte:
    print(data[colonne].value_counts(), "\n")

Gender
Female    2462
Male      2002
Name: count, dtype: int64 

Family_background
Middle class          2642
Lower Middle class    1151
Rich class             671
Name: count, dtype: int64 

Relationship_status
Single               2803
In a Relationship    1457
Complicated           204
Name: count, dtype: int64 

Marital_status
Unmarried    3619
Married       725
Separated     120
Name: count, dtype: int64 

Living_style
Bachelor    2449
Family      2015
Name: count, dtype: int64 

friend_circle
Yes    3215
No     1249
Name: count, dtype: int64 

Health_condition
Normal      1787
Good        1162
Abnormal     881
Critical     634
Name: count, dtype: int64 

Home_division
Dhaka         1527
Khulna         548
Rajshahi       488
Sylhet         445
Chittagong     400
Rangpur        363
Barisal        355
Mymensingh     338
Name: count, dtype: int64 

Depression
No     3263
Yes    1201
Name: count, dtype: int64 



Il y a un peu plus de femmes (2 462) que d'hommes (2 002), la majorité des étudiants sont célibataires et non mariés, et Dhaka est de loin la région la plus représentée. 1 201 étudiants ont répondu Yes à `Depression`.

## Partie 2 — Statistiques de base

In [7]:
data.describe()

,Age,Current_semester,CGPA,Cost_without_study
count,4464.000000,4464.000000,4464.000000,4464.000000
mean,22.619176,5.401210,3.412753,11242.978271
std,2.022980,2.578476,0.385605,6640.144019
min,17.000000,0.000000,2.250000,0.000000
25%,21.000000,3.000000,3.150000,6100.000000
50%,23.000000,5.000000,3.490000,9500.000000
75%,24.000000,7.000000,3.700000,15439.000000
max,28.000000,13.000000,4.000000,32000.000000


In [8]:
colonnes_num = ["Age", "Current_semester", "CGPA", "Cost_without_study"]
data[colonnes_num].median()

Age                     23.00
Current_semester         5.00
CGPA                     3.49
Cost_without_study    9500.00
dtype: float64

`describe()` ne donne les statistiques que pour les colonnes numériques :

- l'âge va de 17 à 28 ans, avec une moyenne de 22,6 ans et une médiane de 23 ans ;
- le semestre va de 0 à 13 (médiane : 5) ;
- la CGPA moyenne est de 3,41, avec un minimum de 2,25 et un maximum de 4 (médiane : 3,49) ;
- le coût hors études varie énormément, de 0 à 32 000 (médiane : 9 500). Je ne connais pas l'unité de cette colonne.

### La CGPA avec numpy

In [9]:
tableau = data["CGPA"].to_numpy()

print("Moyenne avec numpy :", tableau.mean())
print("Moyenne avec pandas :", data["CGPA"].mean())
print("Écart-type avec numpy :", tableau.std())
print("Écart-type avec pandas :", data["CGPA"].std())

Moyenne avec numpy : 3.4127531362007164
Moyenne avec pandas : 3.4127531362007164
Écart-type avec numpy : 0.38556144881787263
Écart-type avec pandas : 0.38560464172102926


Les moyennes sont identiques. Les écarts-types sont très légèrement différents parce que numpy divise par n, alors que pandas divise par n − 1 (écart-type d'un échantillon). Avec 4 464 lignes, la différence est minuscule.

In [10]:
data["Depression"].value_counts(normalize=True)

Depression
No     0.730959
Yes    0.269041
Name: proportion, dtype: float64

26,9 % des étudiants ont répondu Yes à `Depression`.

In [11]:
data["Depression_bin"] = data["Depression"].map({"Yes": 1, "No": 0})
data.head()

,Gender,Age,Family_background,Relationship_status,Marital_status,Living_style,Current_semester,CGPA,friend_circle,Health_condition,Cost_without_study,Home_division,Depression,Depression_bin
0,Female,22,Middle class,Single,Unmarried,Bachelor,8,3.43,Yes,Critical,10113,Chittagong,Yes,1
1,Male,19,Middle class,Single,Unmarried,Bachelor,2,3.70,Yes,Abnormal,13815,Chittagong,Yes,1
2,Male,23,Middle class,Single,Unmarried,Family,8,3.28,No,Abnormal,15450,Dhaka,Yes,1
3,Male,20,Middle class,Complicated,Unmarried,Family,2,3.77,No,Good,17413,Dhaka,No,0
4,Female,21,Middle class,Single,Unmarried,Family,1,4.00,No,Abnormal,16693,Dhaka,Yes,1


J'ajoute une colonne `Depression_bin` qui vaut 1 pour Yes et 0 pour No. Comme elle ne contient que des 0 et des 1, sa moyenne donne directement la proportion de Yes.

In [12]:
print(round(data["Depression_bin"].mean() * 100, 1), "%")

26.9 %


On retrouve bien 26,9 %. Je réutilise cette colonne dans toute la partie 4.

## Partie 3 — Qualité des données

In [13]:
data[colonnes_num].agg(["min", "max"])

,Age,Current_semester,CGPA,Cost_without_study
min,17,0,2.25,0
max,28,13,4.00,32000


L'âge et la CGPA ont des valeurs logiques. Par contre, deux valeurs me paraissent suspectes : un semestre égal à 0 et un coût hors études égal à 0.

In [14]:
print("Étudiants au semestre 0 :", (data["Current_semester"] == 0).sum())
print("Étudiants avec un coût de 0 :", (data["Cost_without_study"] == 0).sum())

Étudiants au semestre 0 : 42
Étudiants avec un coût de 0 : 2


42 étudiants sont au semestre 0 et 2 ont un coût hors études de 0, soit environ 1 % des lignes. Je les garde : le semestre 0 correspond peut-être à des étudiants qui viennent d'arriver, et un coût nul à des étudiants entièrement pris en charge par leur famille. Comme ça représente peu de lignes, ça ne change pas les résultats de cette analyse, mais il faudrait vérifier ce que signifient ces valeurs avant d'entraîner un modèle.